# 01 - Manifesto dos artigos

Este notebook inventaria todos os PDFs em `data/raw/pdf/`, gera IDs estáveis por conteúdo, infere provisoriamente autor/ano pelo nome do arquivo, detecta duplicatas e salva o manifesto em Parquet, Excel, CSV e JSON de resumo.

**Importante:** a ordenação dentro do mesmo ano ainda não é considerada cronologia válida. Se não houver data exata de publicação, artigos do mesmo ano não deverão alimentar o histórico uns dos outros.

In [1]:
from pathlib import Path
import pandas as pd

from wonca_rag.paths import PATHS
from wonca_rag.ingestion.manifest import (
    build_manifest,
    validate_manifest,
    summarize_manifest,
    save_manifest,
)

PDF_ROOT = PATHS.data_root / 'raw' / 'pdf'
OUTPUT_DIR = PATHS.data_root / 'metadata'

print('PDF_ROOT:', PDF_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

PDF_ROOT: C:\Github\wonca-rag\data\raw\pdf
OUTPUT_DIR: C:\Github\wonca-rag\data\metadata


## 1. Conferir quantos PDFs existem

Os arquivos podem estar diretamente em `data/raw/pdf/` ou em subpastas.

In [2]:
pdf_files = sorted(PDF_ROOT.rglob('*.pdf')) if PDF_ROOT.exists() else []
print(f'PDFs encontrados: {len(pdf_files)}')

for path in pdf_files[:20]:
    print('-', path.relative_to(PDF_ROOT))

if len(pdf_files) > 20:
    print(f'... e mais {len(pdf_files) - 20} arquivos.')

PDFs encontrados: 233
- Abramson2005.pdf
- Allen1989.pdf
- Almeida2001.pdf
- Alonso2008.pdf
- Artigos sem resumo\Almeida2001.pdf
- Artigos sem resumo\Barker1989.pdf
- Artigos sem resumo\Starfield1971b (1).pdf
- Artigos sem resumo\Starfield1971c.pdf
- Artigos sem resumo\starfield1973b (1).pdf
- Artigos sem resumo\Starfield1976b.pdf
- Artigos sem resumo\Starfield1981d.pdf
- Artigos sem resumo\Starfield1982c.pdf
- Artigos sem resumo\Starfield1985.pdf
- Artigos sem resumo\Starfield1985b.pdf
- Artigos sem resumo\Starfield1989.pdf
- Artigos sem resumo\Starfield1993.pdf
- Artigos sem resumo\Starfield1993b.pdf
- Artigos sem resumo\starfield1996b.pdf
- Artigos sem resumo\Starfield1996c.pdf
- Artigos sem resumo\Starfield1999.pdf
... e mais 213 arquivos.


## 2. Construir o manifesto

In [3]:
manifest_df = build_manifest(PDF_ROOT)
print('Linhas:', len(manifest_df))
display(manifest_df.head(20))

Linhas: 233


,article_id,filename,relative_path,source_folder,file_size_bytes,sha256,is_duplicate_content,duplicate_group,inferred_first_author,publication_year,publication_date,temporal_group,year_inferred_from_filename,needs_metadata_review
0,ART_E3A2395BE56C,Starfield1967.pdf,Starfield1967.pdf,,339081,e3a2395be56c7e7b3fef9df7fb5524dd7e5236d14ae32f...,False,NaN,Starfield,1967,NaT,1967,True,False
1,ART_8FCDDCD40C31,Starfield1968.pdf,Starfield1968.pdf,,916897,8fcddcd40c316d5087887a45e990d13085ced7a43dedc1...,False,NaN,Starfield,1968,NaT,1968,True,False
2,ART_C31E1FAB9674,Starfield1968b.pdf,Starfield1968b.pdf,,517352,c31e1fab9674f3fb6f7dd939b18f0f90417a761e0a7f03...,False,NaN,Starfield,1968,NaT,1968,True,False
3,ART_EB1086AB18B2,densen1968.pdf,densen1968.pdf,,618306,eb1086ab18b2c10481e59278bbf3a30a520f454443ec46...,False,NaN,densen,1968,NaT,1968,True,False
4,ART_8063B5B4E555,Starfield1969.pdf,Starfield1969.pdf,,723755,8063b5b4e555ef0dc5ff8e9076a1aca201bb7901d6ebe4...,False,NaN,Starfield,1969,NaT,1969,True,False
5,ART_C51FA4EF984E,Starfield1971b (1).pdf,Artigos sem resumo/Starfield1971b (1).pdf,Artigos sem resumo,343727,c51fa4ef984e20dfc6f6fa26e16ccefdd4b4ba622c8ef6...,True,c51fa4ef984e20dfc6f6fa26e16ccefdd4b4ba622c8ef6...,Starfield,1971,NaT,1971,True,False
6,ART_B36F04112348,Starfield1971c.pdf,Artigos sem resumo/Starfield1971c.pdf,Artigos sem resumo,418025,b36f04112348c77bcb47dfdf9973b840d64740b905da0d...,True,b36f04112348c77bcb47dfdf9973b840d64740b905da0d...,Starfield,1971,NaT,1971,True,False
7,ART_CB360E1A11F0,Starfield1971.pdf,Starfield1971.pdf,,1350203,cb360e1a11f0b6fe1b876e5c9eb1f31f37bc68aeed6aec...,False,NaN,Starfield,1971,NaT,1971,True,False
8,ART_C51FA4EF984E,Starfield1971b.pdf,Starfield1971b.pdf,,343727,c51fa4ef984e20dfc6f6fa26e16ccefdd4b4ba622c8ef6...,True,c51fa4ef984e20dfc6f6fa26e16ccefdd4b4ba622c8ef6...,Starfield,1971,NaT,1971,True,False
9,ART_B36F04112348,Starfield1971c.pdf,Starfield1971c.pdf,,418025,b36f04112348c77bcb47dfdf9973b840d64740b905da0d...,True,b36f04112348c77bcb47dfdf9973b840d64740b905da0d...,Starfield,1971,NaT,1971,True,False


## 3. Validar o manifesto

In [4]:
problems = validate_manifest(manifest_df)

if problems:
    print('Problemas encontrados:')
    for problem in problems:
        print('-', problem)
else:
    print('Manifesto válido.')

Manifesto válido.


## 4. Resumo de qualidade

In [5]:
summary = summarize_manifest(manifest_df)
summary_df = pd.DataFrame(
    [(key, value) for key, value in summary.items() if key != 'source_folders'],
    columns=['metric', 'value'],
)
display(summary_df)

print('Arquivos por pasta de origem:')
display(
    pd.DataFrame(
        list(summary['source_folders'].items()),
        columns=['source_folder', 'n_files'],
    )
)

,metric,value
0,n_files,233
1,n_unique_contents,215
2,n_duplicate_files,36
3,n_missing_year,0
4,n_needs_metadata_review,0
5,year_min,1967
6,year_max,2016


Arquivos por pasta de origem:


,source_folder,n_files
0,(root),178
1,Artigos sem resumo,21
2,Editoriais sem resumo,34


## 5. Ver registros que precisam de revisão de metadados

In [6]:
review_df = manifest_df.loc[
    manifest_df['needs_metadata_review'],
    [
        'article_id',
        'filename',
        'relative_path',
        'inferred_first_author',
        'publication_year',
    ],
]

print(f'Precisam de revisão: {len(review_df)}')
display(review_df.head(100))

Precisam de revisão: 0


,article_id,filename,relative_path,inferred_first_author,publication_year


## 6. Conferir duplicatas por conteúdo

In [7]:
duplicates_df = manifest_df.loc[
    manifest_df['is_duplicate_content'],
    ['article_id', 'filename', 'relative_path', 'sha256'],
].sort_values(['sha256', 'relative_path'])

print(f'Arquivos marcados como duplicados: {len(duplicates_df)}')
display(duplicates_df.head(100))

Arquivos marcados como duplicados: 36


,article_id,filename,relative_path,sha256
26,ART_0080057343D5,Starfield1976b.pdf,Artigos sem resumo/Starfield1976b.pdf,0080057343d5757eaf4f35c839a60a5145cfd9efc93680...
27,ART_0080057343D5,Starfield1976b.pdf,Starfield1976b.pdf,0080057343d5757eaf4f35c839a60a5145cfd9efc93680...
197,ART_11226A3608C0,Gervas2008.pdf,Editoriais sem resumo/Gervas2008.pdf,11226a3608c011301d532c4a4a21e1eff817cfeeb9b168...
198,ART_11226A3608C0,Gervas2008.pdf,Gervas2008.pdf,11226a3608c011301d532c4a4a21e1eff817cfeeb9b168...
216,ART_13D4FA27B28A,Berra2011.pdf,Berra2011.pdf,13d4fa27b28add5390ba3f3f46205bb0ce60ab09d3a800...
217,ART_13D4FA27B28A,Berra2011b.pdf,Berra2011b.pdf,13d4fa27b28add5390ba3f3f46205bb0ce60ab09d3a800...
119,ART_3576142A9E5D,Demaeseneer2000.pdf,Demaeseneer2000.pdf,3576142a9e5d38d67ea6c7879c8620ee622c14f3d0b5e2...
120,ART_3576142A9E5D,Demaeseneer2000 (1).pdf,Editoriais sem resumo/Demaeseneer2000 (1).pdf,3576142a9e5d38d67ea6c7879c8620ee622c14f3d0b5e2...
130,ART_404FD0BBD2B8,Almeida2001.pdf,Almeida2001.pdf,404fd0bbd2b8fdf55aaf93fb679af45b3115ad4754624d...
131,ART_404FD0BBD2B8,Almeida2001.pdf,Artigos sem resumo/Almeida2001.pdf,404fd0bbd2b8fdf55aaf93fb679af45b3115ad4754624d...


## 7. Salvar manifesto

In [8]:
if manifest_df.empty:
    raise RuntimeError(
        'Nenhum PDF encontrado. Coloque os PDFs em data/raw/pdf/ e execute novamente.'
    )

result = save_manifest(
    df=manifest_df,
    output_dir=OUTPUT_DIR,
)

print('Arquivos salvos:')
print('-', result.parquet_path)
print('-', result.xlsx_path)
print('-', result.csv_path)
print('-', result.summary_path)

Arquivos salvos:
- C:\Github\wonca-rag\data\metadata\articles_manifest.parquet
- C:\Github\wonca-rag\data\metadata\articles_manifest.xlsx
- C:\Github\wonca-rag\data\metadata\articles_manifest.csv
- C:\Github\wonca-rag\data\metadata\articles_manifest_summary.json


## Resultado esperado

Ao final, `data/metadata/` deverá conter:

- `articles_manifest.parquet`
- `articles_manifest.xlsx`
- `articles_manifest.csv`
- `articles_manifest_summary.json`

Esses arquivos são gerados localmente e estão ignorados pelo Git.